In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone

BRONZE_SCHEMA = "supplysage_bronze"
SILVER_SCHEMA = "supplysage_silver"

SOURCE_TABLE = f"{BRONZE_SCHEMA}.bronze_m5_sell_prices"
TARGET_TABLE = f"{SILVER_SCHEMA}.silver_m5_sell_prices"

TRANSFORM_RUN_ID = f"silver_m5_sell_prices_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
CREATED_BY = "Vigya"
NOTEBOOK_NAME = "08_silver_m5_sell_prices"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

print("TRANSFORM_RUN_ID:", TRANSFORM_RUN_ID)
print("SOURCE_TABLE:", SOURCE_TABLE)
print("TARGET_TABLE:", TARGET_TABLE)

bronze_sell_prices_df = spark.table(SOURCE_TABLE)

print("Bronze sell prices row count:", bronze_sell_prices_df.count())

display(bronze_sell_prices_df.limit(10))
bronze_sell_prices_df.printSchema()

In [0]:
def clean_string_col(col_name):
    return (
        F.when(F.col(col_name).isNull(), None)
        .when(F.trim(F.col(col_name)) == "", None)
        .when(F.lower(F.trim(F.col(col_name))).isin("null", "none", "nan"), None)
        .otherwise(F.trim(F.col(col_name)))
    )


silver_sell_prices_df = (
    bronze_sell_prices_df
    .select(
        clean_string_col("store_id").alias("store_id"),
        clean_string_col("item_id").alias("item_id"),
        F.col("wm_yr_wk").cast("int").alias("wm_year_week"),
        F.col("sell_price").cast("double").alias("sell_price")
    )
    .withColumn("state_id", F.split(F.col("store_id"), "_").getItem(0))
    .withColumn("store_number", F.split(F.col("store_id"), "_").getItem(1).cast("int"))
    .withColumn("price_record_id", F.sha2(F.concat_ws("||", "store_id", "item_id", F.col("wm_year_week").cast("string")), 256))
    .withColumn("is_valid_price", F.col("sell_price").isNotNull() & (F.col("sell_price") > 0))
    .withColumn("silver_transform_run_id", F.lit(TRANSFORM_RUN_ID))
    .withColumn("silver_created_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("silver_created_by", F.lit(CREATED_BY))
    .withColumn("silver_notebook_name", F.lit(NOTEBOOK_NAME))
)

print("Silver sell prices row count:", silver_sell_prices_df.count())

display(silver_sell_prices_df.limit(20))
silver_sell_prices_df.printSchema()

In [0]:
sell_price_validation_rows = []

source_row_count = bronze_sell_prices_df.count()
target_row_count = silver_sell_prices_df.count()

sell_price_validation_rows.append({
    "validation_check": "row_count_preserved",
    "expected_value": str(source_row_count),
    "actual_value": str(target_row_count),
    "status": "PASS" if source_row_count == target_row_count else "FAIL",
    "issue_detail": None if source_row_count == target_row_count else "Silver sell prices row count does not match Bronze row count."
})

null_key_count = (
    silver_sell_prices_df
    .filter(
        F.col("store_id").isNull()
        | F.col("item_id").isNull()
        | F.col("wm_year_week").isNull()
    )
    .count()
)

sell_price_validation_rows.append({
    "validation_check": "business_keys_not_null",
    "expected_value": "0 null key rows",
    "actual_value": str(null_key_count),
    "status": "PASS" if null_key_count == 0 else "FAIL",
    "issue_detail": None if null_key_count == 0 else "Null values found in store_id, item_id, or wm_year_week."
})

invalid_price_count = (
    silver_sell_prices_df
    .filter((F.col("sell_price").isNull()) | (F.col("sell_price") <= 0))
    .count()
)

sell_price_validation_rows.append({
    "validation_check": "sell_price_positive",
    "expected_value": "0 invalid sell_price rows",
    "actual_value": str(invalid_price_count),
    "status": "PASS" if invalid_price_count == 0 else "FAIL",
    "issue_detail": None if invalid_price_count == 0 else "sell_price has null, zero, or negative values."
})

duplicate_grain_count = (
    silver_sell_prices_df
    .groupBy("store_id", "item_id", "wm_year_week")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
    .count()
)

sell_price_validation_rows.append({
    "validation_check": "grain_unique_store_item_week",
    "expected_value": "0 duplicate grain rows",
    "actual_value": str(duplicate_grain_count),
    "status": "PASS" if duplicate_grain_count == 0 else "FAIL",
    "issue_detail": None if duplicate_grain_count == 0 else "Duplicate rows found for store_id + item_id + wm_year_week."
})

invalid_state_count = (
    silver_sell_prices_df
    .filter(~F.col("state_id").isin("CA", "TX", "WI"))
    .count()
)

sell_price_validation_rows.append({
    "validation_check": "state_id_valid",
    "expected_value": "Only CA, TX, WI",
    "actual_value": str(invalid_state_count),
    "status": "PASS" if invalid_state_count == 0 else "FAIL",
    "issue_detail": None if invalid_state_count == 0 else "Unexpected state_id values found."
})

null_price_record_id_count = (
    silver_sell_prices_df
    .filter(F.col("price_record_id").isNull())
    .count()
)

sell_price_validation_rows.append({
    "validation_check": "price_record_id_not_null",
    "expected_value": "0 null price_record_id rows",
    "actual_value": str(null_price_record_id_count),
    "status": "PASS" if null_price_record_id_count == 0 else "FAIL",
    "issue_detail": None if null_price_record_id_count == 0 else "Null price_record_id values found."
})

sell_price_validation_schema = T.StructType([
    T.StructField("validation_check", T.StringType(), True),
    T.StructField("expected_value", T.StringType(), True),
    T.StructField("actual_value", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("issue_detail", T.StringType(), True)
])

sell_price_validation_df = spark.createDataFrame(
    sell_price_validation_rows,
    schema=sell_price_validation_schema
)

display(sell_price_validation_df.orderBy("status", "validation_check"))

fail_count = sell_price_validation_df.filter(F.col("status") == "FAIL").count()

print("Silver sell prices validation failures:", fail_count)

In [0]:
if fail_count == 0:
    (
        silver_sell_prices_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TARGET_TABLE)
    )

    print(f"✅ Wrote Silver sell prices table: {TARGET_TABLE}")
else:
    raise ValueError("Silver sell prices validation failed. Fix issues before writing.")

In [0]:
written_sell_prices_df = spark.table(TARGET_TABLE)

written_row_count = written_sell_prices_df.count()

print("Written Silver sell prices row count:", written_row_count)

display(
    written_sell_prices_df
    .select(
        "price_record_id",
        "store_id",
        "state_id",
        "store_number",
        "item_id",
        "wm_year_week",
        "sell_price",
        "is_valid_price"
    )
    .orderBy("store_id", "item_id", "wm_year_week")
    .limit(20)
)

if written_row_count == 6841121:
    print("✅ Read-back check passed: Silver sell prices table exists and row count is correct.")
else:
    print("❌ Read-back check failed: Row count does not match expected 6,841,121.")

In [0]:
sell_price_validation_results_df = (
    sell_price_validation_df
    .withColumn("transform_run_id", F.lit(TRANSFORM_RUN_ID))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("target_table", F.lit(TARGET_TABLE))
    .withColumn("validated_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("created_by", F.lit(CREATED_BY))
    .withColumn("notebook_name", F.lit(NOTEBOOK_NAME))
    .select(
        "transform_run_id",
        "source_table",
        "target_table",
        "validation_check",
        "expected_value",
        "actual_value",
        "status",
        "issue_detail",
        "validated_at",
        "created_by",
        "notebook_name"
    )
)

validation_results_table = f"{SILVER_SCHEMA}.silver_transform_validation_results"

try:
    spark.sql(f"""
    DELETE FROM {validation_results_table}
    WHERE transform_run_id = '{TRANSFORM_RUN_ID}'
    """)
except Exception as e:
    print("Validation results table does not exist yet. It will be created now.")

(
    sell_price_validation_results_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(validation_results_table)
)

display(
    spark.table(validation_results_table)
    .filter(F.col("transform_run_id") == TRANSFORM_RUN_ID)
    .groupBy("status")
    .agg(F.count("*").alias("validation_check_count"))
    .orderBy("status")
)

print("✅ Notebook 08 PASSED: silver_m5_sell_prices created and validated.")
print("Next notebook: 09_silver_m5_sales_daily")